# 12.13 聽寫一句話，和回答一句話，是同一件事嗎？


有人說：「我今天很累，晚餐做什麼比較省事？」一位助手先把聽到的話寫下來，另一位助手再回答簡單的晚餐建議。第一位的任務是聽寫，通常叫語音辨識或ASR；第二位的任務才是聊天。把原話抄得完全正確，並不等於已回答問題。

前置是[12.8沿時間保留特徵](https://birdhackor.github.io/tiny-perceptron-vlm/12.8.html)與[12.11先轉文字的取捨](https://birdhackor.github.io/tiny-perceptron-vlm/12.11.html)。普通句子比「零到九」長，語音順序更重要。「先去臺北，再去臺中」與相反順序，不應只因含相同地名就得到同一段文字。下面把已取得的頻譜時間框反向排列，看看平均摘要留下什麼。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.natural_concepts import audio_order_report

report = audio_order_report()
print("時間框與頻帶", report["time_frames"], report["bands"])
print("先後序列相同", report["same_time_sequence"])
print("平均摘要近似相同", report["same_mean_with_roundoff_tolerance"])

`audio_order_report()`先接一段440Hz與一段880Hz合成單音，取出21個時間框、每框16個頻帶，再反轉這21框的順序。它把結果放進字典`report`，程式按項目名稱取出框數、頻帶數與兩種比較。輸出21、16、False、True。平均對排列不敏感；浮點加總可能因順序有極小差別，所以這裡比較是否在容許誤差內相同。本例反轉的是已算好的頻譜時間框，沒有使用真人語音，也沒有把原錄音倒放後重新分析。

要實際把普通人說話轉成文字，需要有錄音與逐句稿的資料、可處理順序的模型，並另外留出錄音、盡可能用不同說話者檢查。不同口音、背景聲與專有名詞還要另外測。[20.12](https://birdhackor.github.io/tiny-perceptron-vlm/20.12.html)教你分開核對真人錄音的轉寫與聊天，最後實測集中在[20.13](https://birdhackor.github.io/tiny-perceptron-vlm/20.13.html)；前面的合成單音與英文數字實驗仍只支持各自的窄任務。

練習用一句話說明：哪一種資料教模型「他說了什麼」，哪一種示範教模型「接下來如何回應」？不要求回答一定強，但不要把兩種能力混成一個分數。
